# Student Athletes Economic Distance Pipeline

Clean, reorganized notebook for scraping, merging, and analyzing NCAA athlete data.

---

## Section 1: Setup

Install dependencies, mount Google Drive, and define global constants.

In [3]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
!pip install -q wbgapi thefuzz requests beautifulsoup4 selenium webdriver_manager playwright rapidfuzz
!apt-get update -qq && apt-get install -y -qq chromium-chromedriver chromium-browser > /dev/null 2>&1
!playwright install chromium > /dev/null 2>&1

# ============================================================
# 2. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import os
import re
import time
import string
import subprocess
import requests
from bs4 import BeautifulSoup
from thefuzz import fuzz, process
import wbgapi as wb
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import matplotlib.pyplot as plt

# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

# ============================================================
# 4. GLOBAL CONSTANTS
# ============================================================
BASE_PATH = '/content/drive/MyDrive/RA_Project_EconDistance/'
RAW_DATA_PATH = os.path.join(BASE_PATH, 'raw_data/')
SUCCESS_FILE = os.path.join(RAW_DATA_PATH, 'soccer_rosters.csv')
FAILURE_FILE = os.path.join(RAW_DATA_PATH, 'soccer_roster_failures.csv')

HEADERS = {
    "User-Agent":      "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":         "https://www.tfrrs.org/",
}

DELAY = 3  # seconds between web requests

# Create project directories
for folder in ['raw_data', 'cleaned_data', 'output', 'code']:
    os.makedirs(os.path.join(BASE_PATH, folder), exist_ok=True)

print("Setup complete.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Mounted at /content/drive
Setup complete.


## Section 2: Shared Utilities

Reusable helper functions for Selenium, CSV management, and text cleaning.

In [ ]:
def setup_driver():
    """Configure headless ChromeDriver for Colab."""
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument(
        '--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    return webdriver.Chrome(options=chrome_options)


def append_to_roster_csv(new_df, filepath, key_columns=['school_name', 'name', 'gender']):
    """
    Append new roster data to a CSV, deduplicating on key_columns.

    NOTE: key_columns includes 'gender' to avoid dropping athletes
    who share a name but play on different (men's / women's) teams.
    """
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    column_order = [
        'school_name', 'gender', 'jersey_number', 'name',
        'position', 'class_year', 'height', 'hometown', 'high_school_club'
    ]
    new_df = new_df[[col for col in column_order if col in new_df.columns]]
    if os.path.exists(filepath):
        existing_df = pd.read_csv(filepath)
        combined = pd.concat([existing_df, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=key_columns, keep='last')
    else:
        combined = new_df
    combined.to_csv(filepath, index=False, encoding='utf-8')
    print(f"Saved {len(combined)} rows to: {filepath}")


def clean_text(text):
    """Lowercase, remove punctuation, and collapse whitespace."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join(text.split())
    return text


print("Shared utilities loaded.")

## Section 3: Load IPEDS Data

Load the three IPEDS files (HD2023, EF2023a, SFA2223), fix the BOM issue in HD, and filter to 4-year institutions.

In [ ]:
# Load IPEDS CSV files
file_mapping = {
    'HD2023.csv': 'hd',
    'EF2023a.csv': 'ef',
    'SFA2223.csv': 'sfa',
}

dataframes = {}
for filename, var_name in file_mapping.items():
    file_path = os.path.join(RAW_DATA_PATH, filename)
    df = pd.read_csv(file_path, encoding='latin-1')
    dataframes[var_name] = df
    print(f"Loaded {filename}: {df.shape[0]} rows x {df.shape[1]} cols")

hd  = dataframes['hd']
ef  = dataframes['ef']
sfa = dataframes['sfa']

# Fix BOM character in HD column name
if '\ufeffUNITID' in hd.columns:
    hd.rename(columns={'\ufeffUNITID': 'UNITID'}, inplace=True)
    print("Fixed BOM in HD UNITID column.")
elif 'ï»¿UNITID' in hd.columns:
    hd.rename(columns={'ï»¿UNITID': 'UNITID'}, inplace=True)
    print("Fixed BOM in HD UNITID column.")

# Filter to 4-year institutions (SECTOR 1 = public 4-year, 2 = private nonprofit 4-year)
print(f"\nSECTOR distribution before filtering:")
print(hd['SECTOR'].value_counts().sort_index())

hd = hd[hd['SECTOR'].isin([1, 2])].copy()
print(f"\nAfter filtering to 4-year institutions: {len(hd)} institutions remain.")
print(f"EF shape: {ef.shape}")
print(f"SFA shape: {sfa.shape}")

## Section 4: Load EADA

Load the Equity in Athletics Disclosure Act (EADA) data and merge with IPEDS.

In [ ]:
# Load EADA from nested folder
eada_file_path = os.path.join(
    RAW_DATA_PATH, 'EADA_All_Data_Combined_2022-2023_SAS_SPSS_EXCEL', 'EADA_2023.xlsx'
)
eada = pd.read_excel(eada_file_path)
print(f"Loaded EADA: {eada.shape[0]} rows x {eada.shape[1]} cols")

# Normalize UNITID column name in EADA
if 'unitid' in eada.columns:
    eada.rename(columns={'unitid': 'UNITID'}, inplace=True)

# Ensure UNITID is numeric in both DataFrames
hd['UNITID'] = pd.to_numeric(hd['UNITID'], errors='coerce')
eada['UNITID'] = pd.to_numeric(eada['UNITID'], errors='coerce')
hd.dropna(subset=['UNITID'], inplace=True)
eada.dropna(subset=['UNITID'], inplace=True)

# Merge EADA with HD on UNITID (inner join)
merged_eada_hd = pd.merge(hd, eada, on='UNITID', how='inner', suffixes=('_hd', '_eada'))
print(f"\nInstitutions matched in EADA-HD merge: {merged_eada_hd.shape[0]}")
print("Sample:")
display(merged_eada_hd.head())

## Section 5: WDI Macro Data

Download World Development Indicators from the World Bank API, reshape from wide to long format, and merge into a single macro DataFrame.

In [ ]:
# Download 6 indicators from World Bank API
indicators = {
    'PA.NUS.PPP':       'ppp',
    'PA.NUS.FCRF':      'exchange_rate',
    'SL.UEM.TOTL.ZS':   'unemployment',
    'NY.GDP.PCAP.KD.ZG': 'gdp_growth',
    'NY.GDP.PCAP.PP.KD': 'gdp_per_capita_ppp',
    'PA.NUS.PPPC.RF':    'price_level_ratio',
}

years = range(2010, 2024)

wdi_dataframes = {}
print("Downloading WDI indicators...")

for indicator_code, short_name in indicators.items():
    try:
        df_ind = wb.data.DataFrame(
            indicator_code,
            economy='all',
            time=years,
            labels=True
        )
        wdi_dataframes[short_name] = df_ind
        file_name = f"WDI_{short_name}.csv"
        df_ind.to_csv(os.path.join(RAW_DATA_PATH, file_name), index=False)
        print(f"  Downloaded {indicator_code} -> {short_name} ({df_ind.shape})")
    except Exception as e:
        print(f"  ERROR downloading {indicator_code}: {e}")

print(f"\nDownloaded {len(wdi_dataframes)} indicators.")

In [ ]:
# Reshape each indicator from wide (YR columns) to long format
# Build a country-name-to-code mapping from the first indicator
first_df = list(wdi_dataframes.values())[0]
country_name_to_code = first_df.reset_index().set_index('Country')['economy'].to_dict()
print(f"Country mapping: {len(country_name_to_code)} entries")

reshaped_wdi_dfs = []

for short_name, df_wdi in wdi_dataframes.items():
    df_copy = df_wdi.copy()
    df_copy['country_code'] = df_copy['Country'].map(country_name_to_code).fillna('Unknown')
    df_copy.rename(columns={'Country': 'country_name'}, inplace=True)

    year_cols = [col for col in df_copy.columns if col.startswith('YR')]

    df_long = pd.melt(
        df_copy,
        id_vars=['country_code', 'country_name'],
        value_vars=year_cols,
        var_name='year',
        value_name=short_name
    )
    df_long['year'] = df_long['year'].str.replace('YR', '').astype(int)
    reshaped_wdi_dfs.append(df_long)
    print(f"  Reshaped '{short_name}' -> {df_long.shape}")

print(f"\nReshaped {len(reshaped_wdi_dfs)} DataFrames.")
display(reshaped_wdi_dfs[0].head())

In [ ]:
# Merge all reshaped WDI DataFrames into a single 'macro' DataFrame
print("Merging reshaped WDI DataFrames...")

macro = reshaped_wdi_dfs[0]
for i in range(1, len(reshaped_wdi_dfs)):
    macro = pd.merge(
        macro, reshaped_wdi_dfs[i],
        on=['country_code', 'country_name', 'year'],
        how='outer'
    )
    print(f"  After merging DataFrame {i+1}: {macro.shape}")

print(f"\nFinal macro DataFrame: {macro.shape}")
display(macro.head())

## Section 6: NCAA Directory

Scrape the NCAA member institution directory using the real API with CSRF token.

In [ ]:
# Scrape NCAA directory via the real API endpoint
session = requests.Session()

user_agent = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
)

headers = {
    "User-Agent": user_agent,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

# Step 1: Visit the member listing page to get CSRF token and cookies
print("Fetching CSRF token from NCAA directory page...")
time.sleep(2)

page = session.get(
    "https://web3.ncaa.org/directory/memberList",
    headers=headers
)
print(f"Initial page status: {page.status_code}")

soup = BeautifulSoup(page.text, "html.parser")
csrf_meta = soup.find("meta", {"name": "_csrf"})
csrf_token = csrf_meta["content"] if csrf_meta else None

if csrf_token:
    print(f"CSRF token found: {csrf_token[:20]}...")
else:
    print("WARNING: Could not find CSRF token. API call might fail.")

# Step 2: Call the real API endpoint
api_url = "https://web3.ncaa.org/directory/api/directory/memberList"
api_headers = {
    "User-Agent": user_agent,
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-US,en;q=0.9",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://web3.ncaa.org/directory/memberList?type=1",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-origin",
}
if csrf_token:
    api_headers["X-CSRF-TOKEN"] = csrf_token

params = {"type": "1", "_": str(int(time.time() * 1000))}

print("\nCalling NCAA API...")
time.sleep(2)
response = session.get(api_url, params=params, headers=api_headers)
print(f"API status: {response.status_code}")

if response.status_code != 200:
    raise RuntimeError(f"NCAA API returned {response.status_code}: {response.text[:300]}")

# Step 3: Parse JSON into DataFrame
data = response.json()
print(f"Raw records: {len(data)}")

ncaa = pd.DataFrame(data)

# Select and rename columns
selected_cols = ['nameOfficial', 'division', 'conferenceName', 'memberOrgAddress']
ncaa = ncaa[[c for c in selected_cols if c in ncaa.columns]].copy()
ncaa.rename(columns={
    'nameOfficial': 'name',
    'conferenceName': 'conference',
    'memberOrgAddress': 'address'
}, inplace=True)

# Extract state from nested address dict
if 'address' in ncaa.columns:
    ncaa['state'] = ncaa['address'].apply(
        lambda x: x.get('state') if isinstance(x, dict) else None
    )
    ncaa.drop(columns=['address'], inplace=True)

# Ensure required columns exist
for col in ['name', 'state', 'division', 'conference']:
    if col not in ncaa.columns:
        ncaa[col] = pd.NA
ncaa = ncaa[['name', 'state', 'division', 'conference']]
ncaa = ncaa.dropna(subset=['name', 'state', 'division', 'conference'], how='all').reset_index(drop=True)

# Summary
print(f"\nTotal institutions: {len(ncaa)}")
print("\nBy division:")
display(ncaa['division'].astype(str).value_counts().sort_index())

# Save
save_path = os.path.join(RAW_DATA_PATH, 'ncaa_directory.csv')
ncaa.to_csv(save_path, index=False)
print(f"\nSaved to: {save_path}")

## Section 7: NCAA-IPEDS Crosswalk

Fuzzy-match NCAA school names to IPEDS institution names, then apply manual corrections.

In [ ]:
# Clean name columns for fuzzy matching
ncaa['cleaned_name'] = ncaa['name'].apply(clean_text)
hd['INSTNM_cleaned'] = hd['INSTNM'].apply(clean_text)

# Build a list of (cleaned_name, index) for fast lookup using process.extractOne
hd_choices = hd['INSTNM_cleaned'].tolist()

print("Running fuzzy matching (this may take a few minutes)...")
crosswalk_results = []

for _, row_ncaa in ncaa.iterrows():
    ncaa_name = row_ncaa['cleaned_name']

    # Use process.extractOne for speed (instead of nested loop)
    result = process.extractOne(ncaa_name, hd_choices, scorer=fuzz.token_set_ratio)

    if result:
        best_name, best_score, best_idx = result
        matched_row = hd.iloc[best_idx]
        crosswalk_results.append({
            'ncaa_name':          row_ncaa['name'],
            'ncaa_state':         row_ncaa['state'],
            'ncaa_division':      row_ncaa['division'],
            'ncaa_conference':    row_ncaa['conference'],
            'ipeds_name_matched': matched_row['INSTNM'],
            'ipeds_unitid':       matched_row['UNITID'],
            'match_score':        best_score,
            'cleaned_name':       ncaa_name,
        })

crosswalk_df = pd.DataFrame(crosswalk_results)
print(f"Crosswalk created: {len(crosswalk_df)} entries")
display(crosswalk_df.head())

# Save initial crosswalk
save_path = os.path.join(RAW_DATA_PATH, 'ncaa_ipeds_crosswalk.csv')
crosswalk_df.to_csv(save_path, index=False)
print(f"Saved to: {save_path}")

In [ ]:
# Analyze match score distribution
print("--- Match Score Analysis ---\n")

ranges = [
    (">=95", crosswalk_df['match_score'] >= 95),
    ("90-94", (crosswalk_df['match_score'] >= 90) & (crosswalk_df['match_score'] < 95)),
    ("80-89", (crosswalk_df['match_score'] >= 80) & (crosswalk_df['match_score'] < 90)),
    ("<80",  crosswalk_df['match_score'] < 80),
]

for label, mask in ranges:
    print(f"  Score {label}: {mask.sum()}")

# Show low-scoring matches for review
low_scores = crosswalk_df[crosswalk_df['match_score'] < 85].sort_values('match_score')
if not low_scores.empty:
    print(f"\nLow-scoring matches (<85) for manual review: {len(low_scores)}")
    display(low_scores[['ncaa_name', 'ipeds_name_matched', 'match_score']].head(20))

In [ ]:
# Manual corrections to the crosswalk

# Load crosswalk (in case of kernel restart)
crosswalk = pd.read_csv(os.path.join(RAW_DATA_PATH, 'ncaa_ipeds_crosswalk.csv'))
print(f"Loaded crosswalk: {len(crosswalk)} entries")

# 1. Remove Canadian universities (not in IPEDS)
canadian_schools = ['Simon Fraser University']
removed = crosswalk[crosswalk['ncaa_name'].isin(canadian_schools)]
crosswalk = crosswalk[~crosswalk['ncaa_name'].isin(canadian_schools)]
print(f"\nRemoved {len(removed)} Canadian school(s): {canadian_schools}")

# 2. Simple one-to-one corrections (verified UNITIDs)
simple_corrections = {
    'Manhattan University':                           192703,
    'Manhattanville University':                      192749,
    'Rutgers, The State Univ. of New Jersey, Newark': 186399,
    'The Citadel':                                    217864,
    'University of Pittsburgh, Greensburg':            215275,
    'University of Pittsburgh, Bradford':              215266,
    'University of Pittsburgh, Johnstown':             215284,
    'University of Minnesota, Crookston':              174075,
    'University of Cincinnati':                        201885,
}

for ncaa_name, unitid in simple_corrections.items():
    mask = crosswalk['ncaa_name'] == ncaa_name
    if mask.sum() == 0:
        print(f"  NOT FOUND: '{ncaa_name}'")
        continue
    crosswalk.loc[mask, 'ipeds_unitid'] = unitid
    crosswalk.loc[mask, 'match_score']  = 100
    print(f"  Corrected: {ncaa_name} -> UNITID {unitid}")

# 3. Joint programs -- expand into multiple rows
joint_programs = {
    'Pomona-Pitzer Colleges': [121345, 121257, 123165],
}

joint_rows = []
for ncaa_name, unitids in joint_programs.items():
    original_rows = crosswalk[crosswalk['ncaa_name'] == ncaa_name]
    if len(original_rows) == 0:
        print(f"  NOT FOUND: '{ncaa_name}'")
        continue
    original = original_rows.iloc[0].to_dict()
    for unitid in unitids:
        row = original.copy()
        row['ipeds_unitid'] = unitid
        row['match_score']  = 100
        joint_rows.append(row)
    print(f"  Expanded '{ncaa_name}' into {len(unitids)} rows: {unitids}")

crosswalk = crosswalk[~crosswalk['ncaa_name'].isin(joint_programs.keys())]
crosswalk = pd.concat([crosswalk, pd.DataFrame(joint_rows)], ignore_index=True)

# 4. Check for unmatched schools
unmatched = crosswalk[crosswalk['ipeds_unitid'].isna()]
print(f"\nSchools with no UNITID match: {len(unmatched)}")
if len(unmatched) > 0:
    display(unmatched[['ncaa_name', 'ncaa_state', 'ncaa_division']].sort_values('ncaa_state'))

# 5. Save verified crosswalk
save_path = os.path.join(RAW_DATA_PATH, 'ncaa_ipeds_crosswalk_verified.csv')
crosswalk.to_csv(save_path, index=False)
print(f"\nSaved verified crosswalk to: {save_path} ({len(crosswalk)} entries)")

## Section 8: Platform Detection

Detect the athletics website platform (Sidearm Sports, WMT Digital, PrestoSports) for each school using HTML fingerprinting with confidence scores.

In [ ]:
# Platform fingerprints -- unique HTML/URL signatures per platform

SIDEARM_FINGERPRINTS = [
    'dxbhsrqyrr690.cloudfront.net',   # Sidearm CDN (most reliable)
    'sb.scorecardresearch.com',        # Sidearm always loads this
    'sidearm.nextgen',                 # Sidearm JS bundle
    'sidearmsports',                   # URL or HTML reference
    'sidearm-nav',                     # CSS class
    's-person-card',                   # Roster card class
    '/services/roster.ashx',           # Sidearm API endpoint
]

WMT_FINGERPRINTS = [
    'wmtdigital',                      # Direct reference
    'wmt digital',                     # Footer text
    'partner of wmt',                  # Footer credit
    'imgproxy',                        # WMT uses imgproxy
    'storage.googleapis.com',          # WMT Google Cloud Storage
    'wmt-digital',                     # Hyphenated variant
]

PRESTO_FINGERPRINTS = [
    'prestosports.com',                # Direct URL reference
    'prestoweb',                       # PrestoWeb platform
    'prestostats',                     # PrestoStats reference
    'prestoshots',                     # PrestoShots reference
    '/landing/index',                  # Presto URL pattern
    'presto-sports',                   # Hyphenated variant
    'ps-nav',                          # Presto CSS class
    'ps-roster',                       # Presto roster class
]


def detect_platform(url, html, soup, final_url):
    """
    Multi-signal platform detection.
    Returns (platform, confidence, signals_found).
    """
    url_lower       = url.lower()
    final_url_lower = final_url.lower()
    html_lower      = html.lower()

    meta_content = ' '.join(
        [(tag.get('content') or '') for tag in soup.find_all('meta')]
    ).lower()
    script_srcs = ' '.join(
        [(tag.get('src') or '') + ' ' + (tag.string or '')
         for tag in soup.find_all('script')]
    ).lower()
    link_hrefs = ' '.join(
        [(tag.get('href') or '') for tag in soup.find_all('link')]
    ).lower()

    all_signals = ' '.join([
        url_lower, final_url_lower, html_lower,
        meta_content, script_srcs, link_hrefs
    ])

    sidearm_matches = [fp for fp in SIDEARM_FINGERPRINTS if fp in all_signals]
    wmt_matches     = [fp for fp in WMT_FINGERPRINTS     if fp in all_signals]
    presto_matches  = [fp for fp in PRESTO_FINGERPRINTS   if fp in all_signals]

    scores = {
        'Sidearm Sports': len(sidearm_matches),
        'WMT Digital':    len(wmt_matches),
        'PrestoSports':   len(presto_matches),
    }

    best_platform = max(scores, key=scores.get)
    best_score    = scores[best_platform]

    if best_score == 0:
        return 'Unknown', 'NONE', []

    if best_score >= 3:
        confidence = 'HIGH'
    elif best_score == 2:
        confidence = 'MEDIUM'
    else:
        confidence = 'LOW'

    signals_map = {
        'Sidearm Sports': sidearm_matches,
        'WMT Digital':    wmt_matches,
        'PrestoSports':   presto_matches,
    }
    return best_platform, confidence, signals_map[best_platform]


# --- Load school list from crosswalk or define manually ---
# NOTE: Update this list with your schools' athletics URLs.
# In production, load from a CSV with columns: school_name, url
SCHOOLS = [
    ("University of Virginia",       "https://virginiasports.com/"),
    ("LSU",                          "https://lsusports.net/"),
    ("University of Michigan",       "https://mgoblue.com/"),
    ("University of Florida",        "https://floridagators.com/"),
    ("Stanford University",          "https://gostanford.com/"),
    ("University of North Carolina", "https://goheels.com/"),
    ("Duke University",              "https://goduke.com/"),
    ("Ohio State University",        "https://ohiostatebuckeyes.com/"),
    ("Georgetown University",        "https://guhoyas.com/"),
    ("University of Texas",          "https://texaslonghorns.com/"),
    ("Presbyterian College",         "https://gobluehose.com/"),
    ("Wagner College",               "https://wagnerathletics.com/"),
    ("Sacred Heart University",      "https://sacredheartpioneers.com/"),
]

results = []
print(f"Detecting platforms for {len(SCHOOLS)} schools...\n")

for school_name, url in SCHOOLS:
    try:
        response  = requests.get(url, headers=HEADERS, timeout=20, allow_redirects=True)
        final_url = response.url
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        platform, confidence, signals = detect_platform(
            url, response.text, soup, final_url
        )
        status = 'OK'
        redirect_note = f' -> {final_url}' if final_url.rstrip('/') != url.rstrip('/') else ''
        print(f"  [{confidence}] {school_name:40} -> {platform:15} signals: {signals[:3]}{redirect_note}")

    except Exception as e:
        platform, confidence, signals = 'Error', 'NONE', []
        final_url = url
        status = str(e)[:100]
        print(f"  [ERROR] {school_name:40} -> {status[:60]}")

    results.append({
        'school_name': school_name,
        'url':         final_url,
        'platform':    platform,
        'confidence':  confidence,
        'signals':     ', '.join(signals),
        'status':      status,
    })
    time.sleep(2)

platforms_df = pd.DataFrame(results)

print(f"\nPlatform distribution:")
print(platforms_df['platform'].value_counts().to_string())

# Save HIGH/MEDIUM confidence to production CSV
production_df = platforms_df[
    platforms_df['confidence'].isin(['HIGH', 'MEDIUM'])
][['school_name', 'url', 'platform', 'status']]

save_path = os.path.join(RAW_DATA_PATH, 'school_platforms_base.csv')
if os.path.exists(save_path):
    existing = pd.read_csv(save_path)
    combined = pd.concat([existing, production_df], ignore_index=True)
    combined = combined.drop_duplicates(subset='school_name', keep='last')
    combined.to_csv(save_path, index=False)
    print(f"\nUpdated: {save_path} ({len(combined)} schools)")
else:
    production_df.to_csv(save_path, index=False)
    print(f"\nSaved: {save_path} ({len(production_df)} schools)")

# Low confidence goes to review file
needs_review = platforms_df[
    (platforms_df['confidence'].isin(['LOW', 'NONE'])) |
    (platforms_df['platform'].isin(['Unknown', 'Error']))
]
if len(needs_review) > 0:
    review_path = os.path.join(RAW_DATA_PATH, 'school_platforms_needs_review.csv')
    needs_review.to_csv(review_path, index=False)
    print(f"Review file saved: {review_path}")

## Section 9: Roster URL Expansion

Expand school_platforms_base.csv with men's and women's soccer roster URLs.

In [ ]:
# Expand platform CSV with gender-specific roster URLs
INPUT_FILE  = os.path.join(RAW_DATA_PATH, 'school_platforms_base.csv')
OUTPUT_FILE = os.path.join(RAW_DATA_PATH, 'school_website_platforms.csv')

patterns = {
    'Men':   ['mens-soccer'],
    'Women': ['womens-soccer'],
}


def is_valid_roster_page(url):
    """Check if the URL leads to a valid roster page."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=15, allow_redirects=True)
        if response.status_code != 200:
            return False

        soup = BeautifulSoup(response.content, 'html.parser')

        # Check for roster indicators
        if soup.find('table', class_=re.compile(r'roster', re.I)):
            return True
        if soup.find('ul', class_='sidearm-roster-players'):
            return True
        if soup.find('div', class_=re.compile(r'roster-card|player-card', re.I)):
            return True
        roster_containers = soup.find_all('div', class_=re.compile(r'roster', re.I))
        for container in roster_containers:
            if container.find('a', href=re.compile(r'/roster/')):
                return True
        title = soup.title.string.lower() if soup.title else ''
        if 'roster' in title:
            return True

        return False
    except Exception as e:
        print(f"    Error checking URL: {e}")
        return False


print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)

expanded_rows = []
for idx, row in df.iterrows():
    school   = row['school_name']
    base_url = row['url'].strip().rstrip('/') + '/'
    platform = row['platform']
    status   = row.get('status', 'OK')

    print(f"\n[{idx+1}/{len(df)}] {school} ({platform})")

    for gender, pat_list in patterns.items():
        roster_url = None
        for pattern in pat_list:
            candidate = base_url + f'sports/{pattern}/roster'
            print(f"  Testing {gender}: {candidate}")
            if is_valid_roster_page(candidate):
                roster_url = candidate
                print(f"    Valid roster page found!")
                break
            else:
                print(f"    Not a valid roster page.")

        expanded_rows.append({
            'school_name': school,
            'gender':      gender,
            'platform':    platform,
            'base_url':    base_url,
            'roster_url':  roster_url,
            'status':      status,
        })
        time.sleep(1)

df_expanded = pd.DataFrame(expanded_rows)
df_expanded.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSaved to: {OUTPUT_FILE} ({len(df_expanded)} rows)")
display(df_expanded.head())

## Section 10: TFRRS Scraper

Scrape Division I track & field team URLs from TFRRS, then scrape rosters for each team. Retry failures with Playwright fallback.

In [ ]:
# Scrape D-I team URLs from TFRRS
BASE_TFRRS_URL = "https://www.tfrrs.org"
TFRRS_DI_URL   = "https://www.tfrrs.org/leagues/49.html"

print(f"Fetching: {TFRRS_DI_URL}")
response = requests.get(TFRRS_DI_URL, headers=HEADERS, timeout=30)
print(f"Status: {response.status_code}")
time.sleep(2)

soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table')
if not table:
    raise ValueError("No table found on the TFRRS page!")

rows = table.find_all('tr')[1:]  # skip header
print(f"Table rows: {len(rows)}")

results = []
for row in rows:
    cols = row.find_all('td')
    for col in cols:
        link = col.find('a')
        if not link:
            continue
        href = link.get('href', '')
        if '/teams/tf/' not in href:
            continue

        full_url    = BASE_TFRRS_URL + href if href.startswith('/') else href
        school_name = link.text.strip()

        if '_college_m_' in href:
            gender = 'Men'
        elif '_college_f_' in href:
            gender = 'Women'
        else:
            gender = 'Unknown'

        results.append({
            'school_name':    school_name,
            'gender':         gender,
            'tfrrs_team_url': full_url,
        })

teams_df = pd.DataFrame(results).drop_duplicates(subset='tfrrs_team_url')
teams_df = teams_df.sort_values(['school_name', 'gender']).reset_index(drop=True)

print(f"\nTotal teams: {len(teams_df)}")
print(f"  Men's:   {len(teams_df[teams_df['gender'] == 'Men'])}")
print(f"  Women's: {len(teams_df[teams_df['gender'] == 'Women'])}")

save_path = os.path.join(RAW_DATA_PATH, 'tfrrs_team_urls.csv')
teams_df.to_csv(save_path, index=False)
print(f"\nSaved to: {save_path}")

In [ ]:
# Main TFRRS roster scraper + retry with Playwright fallback

YEAR_MAP = {
    'SR-4': 'Senior',
    'JR-3': 'Junior',
    'SO-2': 'Sophomore',
    'FR-1': 'Freshman',
}


def scrape_tfrrs_roster(url, school_name, gender):
    """
    Scrape the ROSTER table from a TFRRS team page.
    Returns a list of dicts, one per athlete.
    """
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    # Find ROSTER section
    roster_table = None
    for h3 in soup.find_all('h3'):
        if 'ROSTER' in h3.text.upper():
            roster_table = h3.find_next('table')
            break

    if roster_table is None:
        raise ValueError("ROSTER table not found on page")

    athletes = []
    for row in roster_table.find_all('tr')[1:]:  # skip header
        cols = row.find_all('td')
        if len(cols) < 2:
            continue

        name_tag = cols[0].find('a')
        name     = name_tag.text.strip() if name_tag else cols[0].text.strip()
        if not name:
            continue

        profile_url = (
            "https://www.tfrrs.org" + name_tag['href']
            if name_tag and name_tag.get('href') else None
        )

        year_raw   = cols[1].text.strip()
        class_year = YEAR_MAP.get(year_raw, year_raw)

        athletes.append({
            'athlete_name': name,
            'class_year':   class_year,
            'hometown':     None,   # populated later from profile pages
            'event':        None,
            'school_name':  school_name,
            'gender':       gender,
            'sport':        'Track & Field',
            'profile_url':  profile_url,
            'team_url':     url,
        })

    return athletes


def fetch_with_playwright(url):
    """Fetch a URL using Playwright headless browser, returns HTML string."""
    script = f"""
import asyncio
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
        )
        await page.goto("{url}", wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(5000)
        html = await page.content()
        await browser.close()
        with open("/tmp/retry_page.html", "w") as f:
            f.write(html)

asyncio.run(main())
"""
    with open("/tmp/retry_scrape.py", "w") as f:
        f.write(script)

    result = subprocess.run(
        ["python", "/tmp/retry_scrape.py"],
        capture_output=True, text=True, timeout=90
    )
    if result.returncode != 0:
        raise RuntimeError(f"Playwright failed: {result.stderr[:200]}")

    with open("/tmp/retry_page.html", "r") as f:
        return f.read()


def parse_tfrrs_roster_html(html, url, school_name, gender):
    """Parse roster from raw HTML (used for Playwright retries)."""
    soup = BeautifulSoup(html, 'html.parser')
    roster_table = None
    for h3 in soup.find_all('h3'):
        if 'ROSTER' in h3.text.upper():
            roster_table = h3.find_next('table')
            break
    if roster_table is None:
        raise ValueError("ROSTER table not found")

    athletes = []
    for row in roster_table.find_all('tr')[1:]:
        cols = row.find_all('td')
        if len(cols) < 2:
            continue
        name_tag = cols[0].find('a')
        name     = name_tag.text.strip() if name_tag else cols[0].text.strip()
        if not name:
            continue
        profile_url = (
            "https://www.tfrrs.org" + name_tag['href']
            if name_tag and name_tag.get('href') else None
        )
        year_raw   = cols[1].text.strip()
        class_year = YEAR_MAP.get(year_raw, year_raw)
        athletes.append({
            'athlete_name': name,
            'class_year':   class_year,
            'hometown':     None,
            'event':        None,
            'school_name':  school_name,
            'gender':       gender,
            'sport':        'Track & Field',
            'profile_url':  profile_url,
            'team_url':     url,
        })
    if len(athletes) == 0:
        raise ValueError("0 athletes found after parsing")
    return athletes


# --- MAIN SCRAPE LOOP ---
teams_df = pd.read_csv(os.path.join(RAW_DATA_PATH, 'tfrrs_team_urls.csv'))
print(f"Loaded {len(teams_df)} team URLs")

all_athletes   = []
failed_teams   = []
success_count  = 0
total          = len(teams_df)

print(f"\nStarting scrape of {total} teams...\n")

for i, row in teams_df.iterrows():
    url         = row['tfrrs_team_url']
    school_name = row['school_name']
    gender      = row.get('gender', 'Unknown')

    try:
        athletes = scrape_tfrrs_roster(url, school_name, gender)
        if len(athletes) == 0:
            raise ValueError("Roster parsed but 0 athletes found")
        all_athletes.extend(athletes)
        success_count += 1
    except Exception as e:
        failed_teams.append({
            'school_name': school_name,
            'gender':      gender,
            'team_url':    url,
            'error':       str(e),
        })

    teams_done = i + 1
    if teams_done % 10 == 0 or teams_done == total:
        print(f"  {teams_done}/{total} | success: {success_count} | "
              f"failed: {len(failed_teams)} | athletes: {len(all_athletes)}")

    time.sleep(DELAY)

# Save initial results
roster_df   = pd.DataFrame(all_athletes)
failures_df = pd.DataFrame(failed_teams)

print(f"\n{'='*55}")
print(f"INITIAL SCRAPE COMPLETE")
print(f"  Athletes scraped: {len(roster_df)}")
print(f"  Teams succeeded:  {success_count}")
print(f"  Teams failed:     {len(failed_teams)}")

roster_path   = os.path.join(RAW_DATA_PATH, 'tfrrs_all_rosters.csv')
failures_path = os.path.join(RAW_DATA_PATH, 'tfrrs_failures.csv')
roster_df.to_csv(roster_path, index=False)
failures_df.to_csv(failures_path, index=False)
print(f"\nSaved roster to: {roster_path}")
print(f"Saved failures to: {failures_path}")

# --- RETRY FAILURES WITH PLAYWRIGHT ---
if len(failed_teams) > 0:
    print(f"\n--- Retrying {len(failed_teams)} failures with Playwright fallback ---")
    recovered_athletes = []
    still_failed       = []

    for i, row in failures_df.iterrows():
        url         = row['team_url']
        school_name = row['school_name']
        gender      = row.get('gender', 'Unknown')
        print(f"\n[{i+1}/{len(failures_df)}] Retrying: {school_name} ({gender})")

        html = None
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            html = response.text
            print(f"  requests succeeded")
        except Exception as e:
            print(f"  requests failed ({e}) -> trying Playwright...")

        if html is None:
            try:
                html = fetch_with_playwright(url)
                print(f"  Playwright succeeded")
            except Exception as e:
                print(f"  Playwright also failed: {e}")
                still_failed.append({
                    'school_name': school_name, 'gender': gender,
                    'team_url': url, 'error': str(e),
                })
                time.sleep(5)
                continue

        try:
            athletes = parse_tfrrs_roster_html(html, url, school_name, gender)
            recovered_athletes.extend(athletes)
            print(f"  Parsed {len(athletes)} athletes")
        except Exception as e:
            print(f"  Parse failed: {e}")
            still_failed.append({
                'school_name': school_name, 'gender': gender,
                'team_url': url, 'error': f"Parse error: {e}",
            })
        time.sleep(5)

    # Append recovered data
    if recovered_athletes:
        existing  = pd.read_csv(roster_path)
        recovered = pd.DataFrame(recovered_athletes)
        combined  = pd.concat([existing, recovered], ignore_index=True)
        combined.to_csv(roster_path, index=False)
        print(f"\nRoster updated: {len(combined)} total athletes -> {roster_path}")

    still_failed_df = pd.DataFrame(still_failed)
    still_failed_path = os.path.join(RAW_DATA_PATH, 'tfrrs_still_failed.csv')
    still_failed_df.to_csv(still_failed_path, index=False)
    print(f"Still-failed URLs saved: {still_failed_path}")

## Section 11: Soccer Scrapers

Platform-specific scrapers for Sidearm Sports, WMT Digital, and PrestoSports. Then an orchestrator cell dispatches to the right scraper.

In [ ]:
# ============================================================
# SIDEARM SPORTS SCRAPER
# HTML table parsing with dynamic header mapping
# (the version that worked for Michigan)
# ============================================================

def scrape_sidearm_roster(url, school_name, gender):
    """
    Scrape roster data from Sidearm Sports pages.
    Parses HTML tables with dynamic header column mapping.
    """
    req_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    }
    roster_data = []

    try:
        print(f"  Sidearm: {url}")
        response = requests.get(url, headers=req_headers, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        tables = soup.find_all('table')
        for table in tables:
            if 'roster' not in str(table).lower():
                continue

            rows = table.find_all('tr')
            headers_row = rows[0] if rows else None
            headers_text = []
            if headers_row:
                headers_text = [th.get_text(strip=True).lower()
                                for th in headers_row.find_all(['th', 'td'])]

            for row in rows[1:]:
                cols = row.find_all('td')
                if len(cols) < 4:
                    continue

                athlete = {
                    'school_name': school_name,
                    'gender': gender,
                    'jersey_number': '',
                    'name': '',
                    'position': '',
                    'class_year': '',
                    'height': '',
                    'hometown': '',
                    'high_school_club': '',
                }

                class_pattern = (
                    r'\b(Fr\.?|So\.?|Jr\.?|Sr\.?|Gr\.?|Freshman|Sophomore|'
                    r'Junior|Senior|Graduate|R-Fr\.?|R-So\.?|R-Jr\.?|R-Sr\.?|'
                    r'First Year|Second Year|Third Year|Fourth Year)\b'
                )

                for i, col in enumerate(cols):
                    text = col.get_text(strip=True)
                    if i < len(headers_text):
                        header = headers_text[i]
                        if 'no' in header or '#' in header or 'num' in header:
                            athlete['jersey_number'] = text
                        elif 'name' in header or 'player' in header:
                            athlete['name'] = text
                        elif 'pos' in header:
                            athlete['position'] = text
                        elif any(k in header for k in ['year', 'class', 'cl.', 'yr.']):
                            athlete['class_year'] = text
                        elif 'height' in header or 'ht' in header:
                            athlete['height'] = text
                        elif 'hometown' in header or 'city' in header:
                            if '/' in text:
                                parts = text.split('/', 1)
                                athlete['hometown'] = parts[0].strip()
                                athlete['high_school_club'] = parts[1].strip()
                            else:
                                athlete['hometown'] = text
                        elif any(k in header for k in ['school', 'club', 'previous']):
                            athlete['high_school_club'] = text

                    # Fallback: if class_year still empty, check regex
                    if not athlete['class_year'] and re.search(class_pattern, text, re.I):
                        athlete['class_year'] = text

                if athlete['name']:
                    roster_data.append(athlete)

        print(f"  Extracted {len(roster_data)} athletes")

    except Exception as e:
        print(f"  Sidearm error: {e}")

    return pd.DataFrame(roster_data)


print("Sidearm Sports scraper loaded.")

In [ ]:
# ============================================================
# WMT DIGITAL SCRAPER
# Cards + table parsing with Selenium fallback
# FIX: added finally: driver.quit() to prevent resource leaks
# ============================================================

def split_hometown_school(text):
    """Split a 'hometown / high school' string on '/'."""
    if '/' not in text:
        return text, ''
    parts = text.split('/', 1)
    left  = parts[0].strip()
    right = parts[1].strip() if len(parts) > 1 else ''
    if re.search(r'[A-Z][a-zA-Z\s\.\-]+,\s*[A-Z]{2}', left) or \
       re.search(r'[A-Z][a-zA-Z\s\.\-]+,\s*[A-Z][a-z]+', left):
        return left, right
    return text, ''


def extract_compound_line(line):
    """Extract fields from compound lines like height+class+hometown."""
    result = {'height': '', 'class_year': '', 'hometown': '', 'high_school_club': ''}
    temp = line

    height_match = re.search(r'(\d+[\u2032\'\"]\s*\d*[\u2032\'\"]?)', temp)
    if height_match:
        result['height'] = height_match.group(1)
        temp = temp.replace(height_match.group(1), '')

    year_match = re.search(
        r'\b(Redshirt\s+)?(Fr\.?|So\.?|Jr\.?|Sr\.?|Gr\.?|Freshman|Sophomore|'
        r'Junior|Senior|Graduate|Redshirt Freshman|Redshirt Sophomore|'
        r'Redshirt Junior|Redshirt Senior|Graduate Student)\b', temp, re.I
    )
    if year_match:
        result['class_year'] = year_match.group(0).strip()
        temp = temp.replace(year_match.group(0), '')

    hometown_match = re.search(r'Hometown\s*([A-Z][a-zA-Z\s\.\-]+,\s*[A-Z]{2})', temp, re.I)
    if not hometown_match:
        hometown_match = re.search(r'Hometown\s*([A-Z][a-zA-Z\s\.\-]+,\s*[A-Z][a-z]+)', temp, re.I)
    if hometown_match:
        result['hometown'] = hometown_match.group(1).strip()
        temp = temp.replace(hometown_match.group(0), '')

    hs_match = re.search(r'High School\s*([A-Z][a-zA-Z\s\.\-]+)', temp, re.I)
    if not hs_match:
        hs_match = re.search(r'Previous School\s*([A-Z][a-zA-Z\s\.\-]+)', temp, re.I)
    if hs_match:
        result['high_school_club'] = hs_match.group(1).strip()

    return result


def parse_stanford_cards(soup):
    """Parse Stanford's specific card layout."""
    roster_data = []
    cards = soup.select('div.roster-card-item')
    STAFF_KEYWORDS = [
        'coach', 'director', 'assistant', 'manager', 'sports medicine',
        'performance', 'nutrition', 'psychology', 'therapist', 'academic',
        'equipment', 'communications', 'sports performance'
    ]
    height_pattern = r'(\d+[\u2032\'\"]\s*\d*[\u2032\'\"]?)'
    class_pattern = (
        r'\b(Redshirt\s+)?(Fr\.?|So\.?|Jr\.?|Sr\.?|Gr\.?|Freshman|Sophomore|'
        r'Junior|Senior|Graduate|Redshirt Freshman|Redshirt Sophomore|'
        r'Redshirt Junior|Redshirt Senior|Graduate Student)\b'
    )
    hometown_pattern = r'[A-Z][a-zA-Z\s\.\-]+,\s*[A-Z]{2}\.?|[A-Z][a-zA-Z\s\.\-]+,\s*[A-Z][a-z]+'

    for card in cards:
        athlete = {
            'school_name': '', 'jersey_number': '', 'name': '', 'position': '',
            'class_year': '', 'height': '', 'hometown': '', 'high_school_club': ''
        }

        num_elem = card.select_one('strong.roster-card-item__jersey-number')
        if num_elem:
            athlete['jersey_number'] = num_elem.get_text(strip=True)

        img = card.select_one('img')
        if img and img.get('alt'):
            athlete['name'] = img['alt'].strip()
        else:
            name_elem = card.select_one('.roster-card-item__content strong:first-child')
            if name_elem:
                athlete['name'] = name_elem.get_text(strip=True)

        pos_elem = card.select_one('strong.roster-card-item__position')
        if pos_elem:
            athlete['position'] = pos_elem.get_text(strip=True)

        # Skip staff
        if athlete['position'] and any(k in athlete['position'].lower() for k in STAFF_KEYWORDS):
            continue
        if not athlete['name'] or not athlete['position']:
            continue

        card_text = card.get_text(separator='\n', strip=True)
        lines = [line.strip() for line in card_text.split('\n') if line.strip()]

        for line in lines:
            lower = line.lower()
            if 'hometown:' in lower:
                athlete['hometown'] = line.split(':', 1)[-1].strip()
            elif 'high school:' in lower:
                athlete['high_school_club'] = line.split(':', 1)[-1].strip()
            elif 'height:' in lower:
                athlete['height'] = line.split(':', 1)[-1].strip()
            elif 'class:' in lower or 'year:' in lower:
                athlete['class_year'] = line.split(':', 1)[-1].strip()

        if not athlete['height'] or not athlete['class_year'] or not athlete['hometown']:
            for line in lines:
                if not athlete['height'] and re.search(height_pattern, line):
                    athlete['height'] = re.search(height_pattern, line).group(1)
                elif not athlete['class_year'] and re.search(class_pattern, line, re.I):
                    athlete['class_year'] = re.search(class_pattern, line, re.I).group(0)
                elif not athlete['hometown'] and re.search(hometown_pattern, line):
                    athlete['hometown'] = re.search(hometown_pattern, line).group(0)

        if athlete['name']:
            athlete['name'] = re.sub(r'^\d+', '', athlete['name']).strip()
        roster_data.append(athlete)

    return roster_data


def scrape_stanford(url, school_name, gender):
    """Stanford-specific scraper using Selenium."""
    print("  Stanford detected -- using Selenium...")
    driver = setup_driver()
    try:
        driver.get(url)
        time.sleep(5)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        roster_data = parse_stanford_cards(soup)
        for athlete in roster_data:
            athlete['school_name'] = school_name
            athlete['gender'] = gender
        print(f"  Extracted {len(roster_data)} athletes (Stanford)")
        return pd.DataFrame(roster_data)
    except Exception as e:
        print(f"  Selenium error for Stanford: {e}")
        return pd.DataFrame()
    finally:
        driver.quit()


def scrape_generic_wmt(url, school_name, gender):
    """Generic WMT Digital scraper (cards + details extraction)."""
    req_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9',
    }
    try:
        print(f"  WMT (requests): {url}")
        response = requests.get(url, headers=req_headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # Locate player cards
        player_cards = []
        for selector in ['div.roster__player', 'div.roster-card', 'div.player-card', 'li.roster-item']:
            cards = soup.select(selector)
            if cards:
                player_cards = cards
                break

        if not player_cards:
            potential = soup.find_all('div', class_=re.compile(r'roster|player', re.I))
            for div in potential:
                if div.find(string=re.compile(r'#')):
                    player_cards.append(div)

        roster_data = []
        for card in player_cards:
            athlete = {
                'school_name': school_name, 'gender': gender,
                'jersey_number': '', 'name': '', 'position': '',
                'class_year': '', 'height': '', 'hometown': '', 'high_school_club': ''
            }

            # Number
            number_elem = card.find(string=re.compile(r'^#'))
            if number_elem:
                num_text = number_elem if isinstance(number_elem, str) else number_elem.get_text(strip=True)
                match = re.search(r'#(\d+)', num_text)
                if match:
                    athlete['jersey_number'] = match.group(1)

            # Name
            name_elem = card.find(['h3', 'h4'], class_=re.compile(r'name|title', re.I))
            if not name_elem:
                name_elem = card.find(class_=re.compile(r'player-name|full-name', re.I))
            if name_elem:
                raw_name = name_elem.get_text(strip=True)
                athlete['name'] = re.sub(r'Season\s+\d{4}-\d{2}', '', raw_name).strip()

            # Position
            pos_elem = card.find(class_=re.compile(r'position', re.I))
            if pos_elem:
                athlete['position'] = pos_elem.get_text(strip=True)

            # Details from structured elements
            hometown_elem = card.find(class_=re.compile(r'hometown|city', re.I))
            if hometown_elem:
                athlete['hometown'] = hometown_elem.get_text(strip=True)
            hs_elem = card.find(class_=re.compile(r'highschool|high-school|prev-school', re.I))
            if hs_elem:
                athlete['high_school_club'] = hs_elem.get_text(strip=True)

            details_container = card.find(class_=re.compile(r'details|info|bio', re.I))
            if details_container:
                for item in details_container.find_all(['li', 'div', 'p']):
                    text = item.get_text(strip=True)
                    lower = text.lower()
                    if 'hometown:' in lower:
                        athlete['hometown'] = text.split(':', 1)[-1].strip()
                    elif 'high school:' in lower:
                        athlete['high_school_club'] = text.split(':', 1)[-1].strip()
                    elif 'previous school:' in lower:
                        athlete['high_school_club'] = text.split(':', 1)[-1].strip()
                    elif 'height:' in lower:
                        athlete['height'] = text.split(':', 1)[-1].strip()
                    elif 'class:' in lower or 'year:' in lower:
                        athlete['class_year'] = text.split(':', 1)[-1].strip()
                    elif '/' in text and not athlete['hometown']:
                        left, right = split_hometown_school(text)
                        if left != text:
                            athlete['hometown'] = left
                            athlete['high_school_club'] = right

            # Fallback: parse full card text
            if not athlete['hometown'] or not athlete['class_year']:
                card_text = card.get_text(separator='\n', strip=True)
                lines = [l.strip() for l in card_text.split('\n')
                         if l.strip() and l.strip() != athlete['name'] and l.strip() != athlete['position']]
                for line in lines:
                    lower = line.lower()
                    if 'hometown:' in lower and not athlete['hometown']:
                        athlete['hometown'] = line.split(':', 1)[-1].strip()
                    elif 'high school:' in lower and not athlete['high_school_club']:
                        athlete['high_school_club'] = line.split(':', 1)[-1].strip()
                    elif 'previous school:' in lower and not athlete['high_school_club']:
                        athlete['high_school_club'] = line.split(':', 1)[-1].strip()
                    elif 'height:' in lower and not athlete['height']:
                        athlete['height'] = line.split(':', 1)[-1].strip()
                    elif ('class:' in lower or 'year:' in lower) and not athlete['class_year']:
                        athlete['class_year'] = line.split(':', 1)[-1].strip()
                    elif '/' in line and not athlete['hometown']:
                        left, right = split_hometown_school(line)
                        if left != line:
                            athlete['hometown'] = left
                            athlete['high_school_club'] = right
                    elif any(k in line for k in ['lbs', 'Hometown', 'High School', 'Previous School']):
                        compound = extract_compound_line(line)
                        for fld in ['height', 'class_year', 'hometown', 'high_school_club']:
                            if compound[fld] and not athlete[fld]:
                                athlete[fld] = compound[fld]
                    else:
                        if not athlete['height'] and re.search(r'\d+[\u2032\'\"]\s*\d*[\u2032\'\"]?', line):
                            athlete['height'] = re.search(r'\d+[\u2032\'\"]\s*\d*[\u2032\'\"]?', line).group(0)
                        if not athlete['class_year'] and re.search(
                            r'\b(Redshirt\s+)?(Fr|So|Jr|Sr|Gr|Freshman|Sophomore|Junior|Senior|Graduate)\b', line, re.I
                        ):
                            athlete['class_year'] = re.search(
                                r'\b(Redshirt\s+)?(Fr|So|Jr|Sr|Gr|Freshman|Sophomore|Junior|Senior|Graduate)\b', line, re.I
                            ).group(0)
                        if not athlete['hometown'] and re.search(r'[A-Z][a-zA-Z\s\.\-]+,\s*[A-Z]{2}', line):
                            athlete['hometown'] = re.search(r'[A-Z][a-zA-Z\s\.\-]+,\s*[A-Z]{2}', line).group(0)

            # Clean residual labels
            for field in ['hometown', 'high_school_club', 'class_year', 'height']:
                if athlete[field]:
                    athlete[field] = re.sub(
                        r'^(hometown|high school|previous school|class|year|height)\s*:\s*',
                        '', athlete[field], flags=re.I
                    ).strip()

            if athlete['name']:
                roster_data.append(athlete)

        print(f"  Extracted {len(roster_data)} athletes")
        return pd.DataFrame(roster_data)

    except Exception as e:
        print(f"  WMT error: {e}")
        return pd.DataFrame()


def scrape_wmt_roster(url, school_name, gender):
    """WMT Digital dispatcher: Stanford gets special handling."""
    if 'gostanford.com' in url:
        return scrape_stanford(url, school_name, gender)
    else:
        df = scrape_generic_wmt(url, school_name, gender)
        if df is not None and not df.empty:
            return df

        # Selenium fallback for empty results
        print("  Trying Selenium fallback for WMT...")
        driver = setup_driver()
        try:
            driver.get(url)
            time.sleep(5)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            # Try parsing again with JS-rendered content
            player_cards = []
            for selector in ['div.roster__player', 'div.roster-card', 'div.player-card']:
                cards = soup.select(selector)
                if cards:
                    player_cards = cards
                    break
            if player_cards:
                # Re-use the generic parsing logic (simplified)
                roster_data = []
                for card in player_cards:
                    name_elem = card.find(['h3', 'h4'], class_=re.compile(r'name|title', re.I))
                    if name_elem:
                        roster_data.append({
                            'school_name': school_name, 'gender': gender,
                            'name': name_elem.get_text(strip=True),
                        })
                if roster_data:
                    print(f"  Selenium fallback found {len(roster_data)} athletes")
                    return pd.DataFrame(roster_data)
            return pd.DataFrame()
        except Exception as e:
            print(f"  Selenium fallback error: {e}")
            return pd.DataFrame()
        finally:
            driver.quit()


print("WMT Digital scraper loaded.")

In [ ]:
# ============================================================
# PRESTOSPORTS SCRAPER
# Parses sidearm-roster-players list structure
# ============================================================

def scrape_prestosports_roster(url, school_name, gender):
    """
    Scrape roster data from PrestoSports pages.
    Parses the ul.sidearm-roster-players list structure.
    """
    req_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9',
    }
    roster_data = []

    try:
        print(f"  PrestoSports: {url}")
        response = requests.get(url, headers=req_headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        players_container = soup.find('ul', class_='sidearm-roster-players')
        if not players_container:
            print("  Could not find sidearm-roster-players container.")
            return pd.DataFrame()

        players = players_container.find_all('li', class_='sidearm-roster-player')
        print(f"  Found {len(players)} players")

        for player in players:
            athlete = {
                'school_name': school_name, 'gender': gender,
                'jersey_number': '', 'name': '', 'position': '',
                'class_year': '', 'height': '', 'hometown': '', 'high_school_club': ''
            }

            # Jersey number
            num_elem = player.find('span', class_='sidearm-roster-player-jersey-number')
            if num_elem:
                athlete['jersey_number'] = num_elem.get_text(strip=True)

            # Name
            name_elem = player.find('h3')
            if name_elem:
                name_link = name_elem.find('a')
                athlete['name'] = (name_link or name_elem).get_text(strip=True)

            # Position
            pos_elem = player.find('span', class_='text-bold')
            if pos_elem:
                athlete['position'] = pos_elem.get_text(strip=True)

            # Height
            height_elem = player.find('span', class_='sidearm-roster-player-height')
            if height_elem:
                athlete['height'] = height_elem.get_text(strip=True)

            # Class year
            year_elem = player.find('span', class_='sidearm-roster-player-academic-year')
            if year_elem:
                athlete['class_year'] = year_elem.get_text(strip=True)
            else:
                class_div = player.find('div', class_='sidearm-roster-player-class-hometown')
                if class_div:
                    yr_span = class_div.find('span', class_='sidearm-roster-player-academic-year')
                    if yr_span:
                        athlete['class_year'] = yr_span.get_text(strip=True)

            # Hometown and High School
            class_div = player.find('div', class_='sidearm-roster-player-class-hometown')
            if class_div:
                ht_elem = class_div.find('span', class_='sidearm-roster-player-hometown')
                hs_elem = class_div.find('span', class_='sidearm-roster-player-highschool')
                if ht_elem:
                    athlete['hometown'] = ht_elem.get_text(strip=True)
                if hs_elem:
                    athlete['high_school_club'] = hs_elem.get_text(strip=True)

                # Fallback: split full text on '/'
                if not athlete['hometown'] and not athlete['high_school_club']:
                    full_text = class_div.get_text(separator=' ', strip=True)
                    if '/' in full_text:
                        parts = full_text.split('/', 1)
                        athlete['hometown'] = parts[0].strip()
                        athlete['high_school_club'] = parts[1].strip() if len(parts) > 1 else ''
                    else:
                        athlete['hometown'] = full_text
            else:
                ht_elem = player.find('span', class_='sidearm-roster-player-hometown')
                if ht_elem:
                    athlete['hometown'] = ht_elem.get_text(strip=True)
                hs_elem = player.find('span', class_='sidearm-roster-player-highschool')
                if hs_elem:
                    athlete['high_school_club'] = hs_elem.get_text(strip=True)

            # Clean whitespace
            for key in athlete:
                if athlete[key]:
                    athlete[key] = re.sub(r'\s+', ' ', athlete[key]).strip()

            if athlete['name']:
                roster_data.append(athlete)

        print(f"  Extracted {len(roster_data)} athletes")

    except Exception as e:
        print(f"  PrestoSports error: {e}")
        import traceback
        traceback.print_exc()

    return pd.DataFrame(roster_data)


print("PrestoSports scraper loaded.")

In [ ]:
# ============================================================
# SOCCER SCRAPER ORCHESTRATOR
# Loads school_website_platforms.csv and dispatches to the
# correct scraper based on platform column.
# ============================================================

PLATFORM_MAP = {
    'Sidearm Sports': scrape_sidearm_roster,
    'WMT Digital':    scrape_wmt_roster,
    'PrestoSports':   scrape_prestosports_roster,
}

INPUT_FILE = os.path.join(RAW_DATA_PATH, 'school_website_platforms.csv')
print(f"Loading {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Total rosters to process: {len(df)}")

all_data = []
failures = []

for idx, row in df.iterrows():
    school     = row['school_name']
    gender     = row.get('gender', '')
    platform   = row['platform']
    roster_url = row['roster_url']

    # Skip rows without a valid URL
    if pd.isna(roster_url) or not str(roster_url).startswith('http'):
        print(f"[{idx+1}/{len(df)}] {school} ({gender}) - No valid URL")
        failures.append({
            'school': school, 'gender': gender,
            'reason': 'No roster URL', 'url': roster_url
        })
        continue

    print(f"\n[{idx+1}/{len(df)}] {school} ({gender}) - {platform}")
    print(f"   URL: {roster_url}")

    if platform not in PLATFORM_MAP:
        print(f"   Unknown platform: {platform}")
        failures.append({
            'school': school, 'gender': gender,
            'reason': f'Unknown platform: {platform}', 'url': roster_url
        })
        time.sleep(DELAY)
        continue

    scraper   = PLATFORM_MAP[platform]
    df_result = scraper(roster_url, school, gender)

    if df_result is not None and not df_result.empty:
        all_data.append(df_result)
        print(f"   Success: {len(df_result)} athletes")
    else:
        print(f"   No data returned")
        failures.append({
            'school': school, 'gender': gender,
            'reason': 'No data found', 'url': roster_url
        })

    time.sleep(DELAY)

# Save results
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv(SUCCESS_FILE, index=False, encoding='utf-8')
    print(f"\nSaved {len(final_df)} records to {SUCCESS_FILE}")
else:
    print("\nNo data collected.")

if failures:
    fail_df = pd.DataFrame(failures)
    fail_df.to_csv(FAILURE_FILE, index=False, encoding='utf-8')
    print(f"Saved {len(fail_df)} failures to {FAILURE_FILE}")

## Section 12: TFRRS Hometown Scraper (NEW)

Scrape individual TFRRS athlete profile pages to extract hometown data. Updates the existing tfrrs_all_rosters.csv.

In [5]:
# Explicitly install rapidfuzz in a new cell
!pip install rapidfuzz

In [12]:
# Re-import necessary libraries after installation
import pandas as pd
import numpy as np
import os
import re
import time
import string
import subprocess
import requests
from bs4 import BeautifulSoup
from thefuzz import fuzz, process
import wbgapi as wb
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import matplotlib.pyplot as plt

# Re-import rapidfuzz explicitly after installation
from rapidfuzz import fuzz, process

In [ ]:
"""
============================================================
NCAA TRACK & FIELD ROSTER COUNTRY ENRICHMENT
============================================================

Reads an URL discovery template (url_discovery_template.xlsx) where the RA
has filled in athletic department roster URLs for ~355 schools, scrapes each
roster page, parses hometown, and joins the resulting country/intl flag
back to the original 25k-athlete TFRRS file.

OUTPUT
------
    enriched_athletes.csv      -- original 25k rows + city, state, country,
                                  is_international columns
    enrichment_log.csv         -- per-school: athletes seen on TFRRS,
                                  athletes matched on athletic site, match rate
    schools_unmatched.csv      -- schools where coverage is < 50%
                                  (manual follow-up list)

USAGE
-----
    # 1. After RA fills in url_discovery_template.xlsx:
    python enrich_country.py \
        --tfrrs    All_Rosters_Data.csv \
        --template url_discovery_template_filled.xlsx \
        --out-dir  ./enrichment_output

DEPENDENCIES
------------
    pip install requests beautifulsoup4 pandas lxml openpyxl rapidfuzz playwright nest_asyncio
    playwright install chromium
"""
import argparse
import hashlib
import json
import logging
import os
import random
import re
import time
import subprocess
import asyncio
import nest_asyncio
nest_asyncio.apply()
from pathlib import Path
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from rapidfuzz import fuzz, process
from playwright.async_api import async_playwright

# ============================================================
# Config
# ============================================================
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (academic research; Florida Tech College of Business; "
        "contact: amalkova@fit.edu) NCAA-roster-collection/1.0"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
DELAY_RANGE = (2.5, 4.5)
TIMEOUT = 30
MIN_ROSTER_SIZE = 3
NAME_MATCH_THRESHOLD = 85   # rapidfuzz score 0-100

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("enrich")


# ============================================================
# robots.txt (cached per host)
# ============================================================
_robots_cache = {}

def is_allowed(url):
    """
    Check robots.txt. Fails OPEN on network errors (403 from proxies, timeouts,
    hosts that don't serve robots.txt) — these are transport issues, not
    policy signals. Only fails CLOSED when we successfully read a robots.txt
    and it explicitly disallows the URL.
    """
    p = urlparse(url)
    root = f"{p.scheme}://{p.netloc}"
    if root not in _robots_cache:
        try:
            r = requests.get(root + "/robots.txt", timeout=10, headers=HEADERS)
            if r.status_code == 200:
                rp = RobotFileParser()
                rp.parse(r.text.splitlines())
                _robots_cache[root] = rp
            else:
                # 403/404/500 — can't read the file, assume allowed
                log.info(f"  robots.txt returned {r.status_code} for {root}; "
                         f"proceeding (fail-open).")
                _robots_cache[root] = None
        except Exception as e:
            log.info(f"  robots.txt unreachable for {root}: {e}; "
                     f"proceeding (fail-open).")
            _robots_cache[root] = None
    rp = _robots_cache[root]
    return True if rp is None else rp.can_fetch(HEADERS["User-Agent"], url)


# ============================================================
# Fetching with HTML cache + Playwright fallback
# ============================================================
def _cache_path(cache_dir, url):
    h = hashlib.md5(url.encode()).hexdigest()[:12]
    safe = re.sub(r"[^A-Za-z0-9]+", "_", url)[:80]
    return Path(cache_dir) / f"{safe}_{h}.html"


def fetch_html(url, cache_dir, use_browser=True):
    cp = _cache_path(cache_dir, url)
    if cp.exists():
        return cp.read_text(encoding="utf-8"), "cache"

    if not is_allowed(url):
        log.warning(f"  robots.txt DISALLOWS {url}")
        return None, "blocked"

    html = ""
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
        html = r.text
    except Exception as e:
        log.warning(f"  requests failed: {e}")

    looks_empty = (not html) or "sidearm-roster-player" not in html.lower()
    if looks_empty and use_browser:
        log.info("  trying browser fallback...")
        html = fetch_with_browser(url)

    if html:
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(html, encoding="utf-8")
    return html, ("requests" if not looks_empty else "browser")


def fetch_with_browser(url):
    async def _run_playwright():
        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch(
                    headless=True,
                    args=["--no-sandbox", "--disable-dev-shm-usage"]
                )
                ctx = await browser.new_context(user_agent=HEADERS['User-Agent'])
                page = await ctx.new_page()
                await page.goto(url, wait_until="networkidle", timeout=45000)
                try:
                    await page.wait_for_selector(
                        "li.sidearm-roster-player, tr.sidearm-roster-player, [class*='roster-player']",
                        timeout=10000,
                    )
                except Exception:
                    pass
                html = await page.content()
                await browser.close()
                return html
        except Exception as e:
            log.warning(f"  browser fetch failed internally: {e}")
            return ""

    try:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(_run_playwright())
    except Exception as e:
        log.warning(f"  browser fetch failed: {e}")
        return ""


# ============================================================
# Parsing (Sidearm card + table layouts)
# ============================================================
def _clean(s):
    return re.sub(r"\s+", " ", s).strip() if s else ""


def _grab(card, key):
    pat = re.compile(rf"sidearm-roster-player-{re.escape(key)}\b", re.I)
    el = card.find(class_=pat)
    return _clean(el.get_text(" ", strip=True)) if el else ""


def parse_sidearm_roster(html):
    if not html:
        return []
    soup = BeautifulSoup(html, "lxml")
    players = _parse_card_layout(soup)
    if len(players) < MIN_ROSTER_SIZE:
        table_players = _parse_table_layout(soup)
        if len(table_players) > len(players):
            players = table_players
    return players


def _parse_card_layout(soup):
    out, seen = [], set()
    cards = soup.find_all(class_="sidearm-roster-player")
    if not cards:
        cards = soup.select(
            "ul.sidearm-roster-players > li, div.sidearm-roster-players > div"
        )
    for card in cards:
        rec = {
            "name":     _grab(card, "name"),
            "hometown": _grab(card, "hometown"),
        }
        if not rec["name"]:
            continue
        key = (rec["name"].lower(), rec["hometown"].lower())
        if key in seen:
            continue
        seen.add(key)
        out.append(rec)
    return out


def _parse_table_layout(soup):
    out = []
    for table in soup.find_all("table"):
        headers = [_clean(th.get_text()).lower()
                   for th in (table.select("thead th") or table.find_all("th"))]
        if not headers or not any("hometown" in h for h in headers):
            continue
        col = {h: i for i, h in enumerate(headers)}

        def at(cells, *aliases):
            for a in aliases:
                for h, i in col.items():
                    if a in h and i < len(cells):
                        return _clean(cells[i].get_text(" ", strip=True))
            return ""

        for tr in table.select("tbody tr"):
            cells = tr.find_all(["td", "th"])
            if len(cells) < len(headers) - 1:
                continue
            rec = {
                "name":     at(cells, "name", "athlete"),
                "hometown": at(cells, "hometown"),
            }
            if rec["name"]:
                out.append(rec)
        if out:
            break
    return out


# ============================================================
# Hometown -> country
# ============================================================
US_STATES = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA",
    "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
    "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT",
    "VA","WA","WV","WI","WY","DC",
}
AP_STATE = {
    "ala":"AL","alaska":"AK","ariz":"AZ","ark":"AR","calif":"CA","colo":"CO",
    "conn":"CT","del":"DE","fla":"FL","ga":"GA","hawaii":"HI","idaho":"ID",
    "ill":"IL","ind":"IN","iowa":"IA","kan":"KS","kans":"KS","ky":"KY","la":"LA",
    "maine":"ME","md":"MD","mass":"MA","mich":"MI","minn":"MN","miss":"MS",
    "mo":"MO","mont":"MT","neb":"NE","nebr":"NE","nev":"NV","nh":"NH","nj":"NJ",
    "nm":"NM","nmex":"NM","ny":"NY","nc":"NC","nd":"ND","ndak":"ND",
    "ohio":"OH","okla":"OK","ore":"OR","oreg":"OR","pa":"PA","ri":"RI","sc":"SC",
    "sd":"SD","sdak":"SD","tenn":"TN","texas":"TX","utah":"UT","vt":"VT",
    "va":"VA","wash":"WA","wva":"WV","wis":"WI","wisc":"WI","wyo":"WY","dc":"DC",
}
STATE_FULL = {
    "alabama":"AL","alaska":"AK","arizona":"AZ","arkansas":"AR","california":"CA",
    "colorado":"CO","connecticut":"CT","delaware":"DE","florida":"FL","georgia":"GA",
    "hawaii":"HI","idaho":"ID","illinois":"IL","indiana":"IN","iowa":"IA",
    "kansas":"KS","kentucky":"KY","louisiana":"LA","maine":"ME","maryland":"MD",
    "massachusetts":"MA","michigan":"MI","minnesota":"MN","mississippi":"MS",
    "missouri":"MO","montana":"MT","nebraska":"NE","nevada":"NV","new hampshire":"NH",
    "new jersey":"NJ","new mexico":"NM","new york":"NY","north carolina":"NC",
    "north dakota":"ND","ohio":"OH","oklahoma":"OK","oregon":"OR","pennsylvania":"PA",
    "rhode island":"RI","south carolina":"SC","south dakota":"SD","tennessee":"TN",
    "texas":"TX","utah":"UT","vermont":"VT","virginia":"VA","washington":"WA",
    "west virginia":"WV","wisconsin":"WI","wyoming":"WY",
    "district of columbia":"DC","washington d.c.":"DC","washington dc":"DC",
}
US_COUNTRY = {"usa","u.s.a","us","u.s","united states","united states of america",
              "america"}


def _state_token(tok):
    if not tok:
        return None
    t = tok.strip().rstrip(".").strip()
    if not t:
        return None
    if t.upper() in US_STATES:
        return t.upper()
    key = t.lower().replace(".", "").strip()
    if key in AP_STATE:
        return AP_STATE[key]
    return STATE_FULL.get(t.lower())


def parse_hometown(ht):
    """Return (city, state, country, is_international)."""
    if not ht or not isinstance(ht, str):
        return ("", "", "", None)
    s = re.sub(r"^\s*hometown\s*[:\-]?\s*", "", ht, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip()
    if s.endswith(".") and len(s.split(",")[-1].strip().rstrip(".")) > 2:
        s = s.rstrip(".")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if not parts:
        return ("", "", "", None)
    city = parts[0]
    if len(parts) == 2:
        st = _state_token(parts[1])
        if st:
            return (city, st, "USA", False)
        return (city, "", parts[1].rstrip("."), True)
    if len(parts) >= 3:
        country = parts[-1].rstrip(".")
        region = parts[-2]
        st = _state_token(country)
        if st:
            return (city, st, "USA", False)
        if country.lower() in US_COUNTRY:
            st = _state_token(region)
            return (city, st or region, "USA", False)
        return (city, region, country, True)
    return (city, "", "", None)


# ============================================================
# Name matching (TFRRS "Last, First" <-> Sidearm "First Last")
# ============================================================
def normalize_name(s):
    if not s or not isinstance(s, str):
        return ""
    s = s.lower().strip()
    if "," in s:
        last, _, first = s.partition(",")
        s = f"{first.strip()} {last.strip()}"
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # Drop common suffixes
    s = re.sub(r"\b(jr|sr|ii|iii|iv)\b", "", s).strip()
    return s


def match_athletes(tfrrs_names, scraped_records):
    """
    For each TFRRS name, find best fuzzy match among scraped records.
    Returns dict: tfrrs_name -> scraped_record (or None).
    """
    if not scraped_records:
        return {n: None for n in tfrrs_names}
    scraped_norm = {normalize_name(r["name"]): r for r in scraped_records}
    keys = list(scraped_norm.keys())
    out = {}
    for tn in tfrrs_names:
        tn_norm = normalize_name(tn)
        if not tn_norm:
            out[tn] = None
            continue
        match = process.extractOne(tn_norm, keys, scorer=fuzz.token_sort_ratio)
        if match and match[1] >= NAME_MATCH_THRESHOLD:
            out[tn] = scraped_norm[match[0]]
        else:
            out[tn] = None
    return out


# ============================================================
# Driver
# ============================================================
SKIP_VALUES = {"", "PRIVATE", "NO_TEAM", "NOT_FOUND", "NA", "N/A"}


def load_template(path):
    """Load the URL template, return list of (school, gender, url) tasks."""
    df = pd.read_csv(path)
    tasks = []
    for _, row in df.iterrows():
        school = row["school_name"]
        gender = row.get("gender", "Unknown")
        url = str(row.get("tfrrs_team_url", "")).strip()
        if url and url.upper() not in SKIP_VALUES and url.startswith("http"):
            tasks.append((school, gender, url))
    return tasks


def main(args):
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = out_dir / "html_cache"

    # Load inputs
    tfrrs = pd.read_csv(args.tfrrs)
    log.info(f"Loaded {len(tfrrs):,} TFRRS athletes across "
             f"{tfrrs['school_name'].nunique()} schools")

    tasks = load_template(args.template)
    log.info(f"Loaded {len(tasks)} (school, gender, url) tasks from template")

    # Per-school enrichment
    enriched_records = []   # list of dicts to merge into tfrrs
    log_rows = []

    for i, (school, gender, url) in enumerate(tasks, start=1):
        log.info(f"[{i}/{len(tasks)}] {school} ({gender}): {url}")

        tfrrs_subset = tfrrs[(tfrrs["school_name"] == school) &
                             (tfrrs["gender"] == gender)]
        if tfrrs_subset.empty:
            log.warning(f"  no TFRRS athletes for this school+gender; skipping")
            continue

        html, src = fetch_html(url, cache_dir, use_browser=args.use_browser)
        scraped = parse_sidearm_roster(html) if html else []
        log.info(f"  parsed {len(scraped)} athletes from page (source: {src})")

        # Match TFRRS names -> scraped records
        tfrrs_names = tfrrs_subset["athlete_name"].tolist()
        matches = match_athletes(tfrrs_names, scraped)

        n_matched = 0
        for tname in tfrrs_names:
            rec = matches.get(tname)
            if rec is None:
                continue
            n_matched += 1
            city, state, country, intl = parse_hometown(rec["hometown"])
            enriched_records.append({
                "athlete_name": tname,
                "school_name":  school,
                "gender":       gender,
                "scraped_name": rec["name"],
                "hometown":     rec["hometown"],
                "city":         city,
                "state":        state,
                "country":      country,
                "is_international": intl,
                "source_url":   url,
            })
        match_rate = n_matched / len(tfrrs_subset) if len(tfrrs_subset) else 0
        log.info(f"  matched {n_matched}/{len(tfrrs_subset)} athletes "
                 f"({100*match_rate:.0f}%)")
        log_rows.append({
            "school": school, "gender": gender, "url": url,
            "tfrrs_n": len(tfrrs_subset), "scraped_n": len(scraped),
            "matched_n": n_matched, "match_rate": match_rate,
            "fetch_source": src,
        })

        # Checkpoint every 25 tasks
        if i % 25 == 0:
            pd.DataFrame(enriched_records).to_csv(
                out_dir / "enriched_athletes_partial.csv", index=False)
            pd.DataFrame(log_rows).to_csv(
                out_dir / "enrichment_log_partial.csv", index=False)
            log.info(f"  [CHECKPOINT] saved partial results")

        time.sleep(random.uniform(*DELAY_RANGE))

    # Final outputs
    enriched_df = pd.DataFrame(enriched_records)
    log_df = pd.DataFrame(log_rows)

    # Merge into the original 25k file. Handle the empty case safely.
    if enriched_df.empty:
        log.warning("No athletes matched. Writing the input file unchanged "
                    "with empty enrichment columns.")
        final = tfrrs.copy()
        for col in ["city", "state", "country", "is_international"]:
            final[col] = pd.NA
    else:
        final = tfrrs.merge(
            enriched_df[["athlete_name", "school_name", "gender",
                         "city", "state", "country", "is_international"]],
            on=["athlete_name", "school_name", "gender"],
            how="left",
            suffixes=("", "_scraped"),
        )

    final.to_csv(out_dir / "enriched_athletes.csv", index=False)
    log_df.to_csv(out_dir / "enrichment_log.csv", index=False)

    # Schools needing manual follow-up
    if not log_df.empty:
        poor = log_df[log_df["match_rate"] < 0.5]
        poor.to_csv(out_dir / "schools_unmatched.csv", index=False)
    else:
        poor = pd.DataFrame()

    # Summary
    n_total = len(final)
    n_country = final["country"].notna().sum()
    n_intl = (final["is_international"] == True).sum()
    log.info("=" * 60)
    log.info(f"DONE.")
    log.info(f"  athletes total:           {n_total:,}")
    log.info(f"  athletes with country:    {n_country:,} ({100*n_country/n_total:.1f}%)")
    log.info(f"  flagged international:    {n_intl:,} ({100*n_intl/n_total:.1f}%)")
    log.info(f"  schools w/ <50% match:    {len(poor)}")
    log.info(f"  outputs in:               {out_dir}")


if __name__ == "__main__":
    # In a Colab environment, command-line arguments are not passed directly.
    # We need to define them programmatically or use values from the notebook state.

    # Ensure RAW_DATA_PATH is defined (from previous notebook cells)
    if 'RAW_DATA_PATH' not in globals() or 'BASE_PATH' not in globals():
        print("RAW_DATA_PATH or BASE_PATH is not defined. Please run Section 1: Setup.")
    else:
        # Define the required paths using BASE_PATH from the notebook setup
        tfrrs_file = os.path.join(RAW_DATA_PATH, 'tfrrs_all_rosters.csv')
        # Use tfrrs_team_urls.csv instead of the missing url_discovery_template_filled.xlsx
        template_file = os.path.join(RAW_DATA_PATH, 'tfrrs_team_urls.csv')
        output_directory = os.path.join(BASE_PATH, 'enrichment_output')

        # Create a Namespace object to simulate argparse.parse_args()
        args = argparse.Namespace(
            tfrrs=tfrrs_file,
            template=template_file,
            out_dir=output_directory,
            use_browser=True  # Set to True to enable Playwright fallback
        )
        main(args)


Browser logs:

<launching> /root/.cache/ms-playwright/chromium_headless_shell-1208/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-commit-input --disable-hang-monitor --disable-ipc-flooding-protection --disable-popup-blocking --disa

In [18]:
import os

# Assuming RAW_DATA_PATH is defined from Section 1: Setup
if 'RAW_DATA_PATH' in globals():
    print(f"Files in {RAW_DATA_PATH}:")
    try:
        for file in os.listdir(RAW_DATA_PATH):
            print(f"- {file}")
    except FileNotFoundError:
        print(f"Error: Directory not found at {RAW_DATA_PATH}")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print("RAW_DATA_PATH is not defined. Please run Section 1: Setup first.")

Files in /content/drive/MyDrive/RA_Project_EconDistance/raw_data/:
- tfrrs_team_urls.csv
- tfrrs_failures.csv
- tfrrs_still_failed.csv
- tfrrs_all_rosters.csv


In [15]:
import os

file_to_check = os.path.join(RAW_DATA_PATH, 'url_discovery_template_filled.xlsx')
if os.path.exists(file_to_check):
    print(f"The file '{file_to_check}' exists.")
else:
    print(f"The file '{file_to_check}' does NOT exist. Please ensure it is uploaded to this directory.")

The file '/content/drive/MyDrive/RA_Project_EconDistance/raw_data/url_discovery_template_filled.xlsx' does NOT exist. Please ensure it is uploaded to this directory.


In [ ]:
# ============================================================
# TFRRS HOMETOWN SCRAPER
# Scrapes individual athlete profile pages for hometown data.
# ============================================================

def scrape_tfrrs_hometown(profile_url):
    """
    Scrape hometown from a TFRRS athlete profile page.

    TFRRS profile pages typically have athlete bio info near the top,
    including hometown in a structured format.

    Returns: hometown string or None
    """
    try:
        response = requests.get(profile_url, headers=HEADERS, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Strategy 1: Look for a div/span with "Hometown" label
        # TFRRS uses a panel-body or similar section with athlete details
        for tag in soup.find_all(['div', 'span', 'td', 'p']):
            text = tag.get_text(strip=True)
            if text.lower().startswith('hometown'):
                # The hometown value is often the next sibling or after a colon
                hometown = text.split(':', 1)[-1].strip() if ':' in text else ''
                if hometown:
                    return hometown
                # Check next sibling
                nxt = tag.find_next_sibling()
                if nxt:
                    return nxt.get_text(strip=True)

        # Strategy 2: Look for "hometown" in a table row
        for row in soup.find_all('tr'):
            cells = row.find_all(['td', 'th'])
            for i, cell in enumerate(cells):
                if 'hometown' in cell.get_text(strip=True).lower():
                    if i + 1 < len(cells):
                        return cells[i + 1].get_text(strip=True)

        # Strategy 3: Look in the page header / panel area
        # Some TFRRS pages have the hometown in a panel-body with athlete info
        panel = soup.find('div', class_=re.compile(r'panel-body|athlete-info|bio', re.I))
        if panel:
            panel_text = panel.get_text(separator='\n', strip=True)
            for line in panel_text.split('\n'):
                if 'hometown' in line.lower():
                    return line.split(':', 1)[-1].strip() if ':' in line else ''

        return None

    except Exception as e:
        return None


# Load the roster data
roster_path = os.path.join(RAW_DATA_PATH, 'tfrrs_all_rosters.csv')
roster_df = pd.read_csv(roster_path)
print(f"Loaded {len(roster_df)} athletes from tfrrs_all_rosters.csv")

# Get unique profile URLs (skip None/NaN)
profile_urls = roster_df.dropna(subset=['profile_url'])['profile_url'].unique()
print(f"Unique profile URLs to scrape: {len(profile_urls)}")

# Skip profiles where we already have hometown data
already_have = roster_df.dropna(subset=['hometown'])
if len(already_have) > 0:
    already_urls = set(already_have['profile_url'].dropna().unique())
    profile_urls = [u for u in profile_urls if u not in already_urls]
    print(f"Skipping {len(already_urls)} already-scraped profiles")
    print(f"Remaining: {len(profile_urls)}")

# Scrape hometown for each profile
hometown_map = {}  # profile_url -> hometown
errors = 0

for i, url in enumerate(profile_urls):
    hometown = scrape_tfrrs_hometown(url)
    if hometown:
        hometown_map[url] = hometown

    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(profile_urls)} | "
              f"found: {len(hometown_map)} | errors: {errors}")

    # Periodic save every 100 athletes
    if (i + 1) % 100 == 0:
        for purl, ht in hometown_map.items():
            mask = roster_df['profile_url'] == purl
            roster_df.loc[mask, 'hometown'] = ht
        roster_df.to_csv(roster_path, index=False)
        print(f"  [CHECKPOINT] Saved progress at {i+1} profiles")

    time.sleep(DELAY)

# Final update and save
for purl, ht in hometown_map.items():
    mask = roster_df['profile_url'] == purl
    roster_df.loc[mask, 'hometown'] = ht

roster_df.to_csv(roster_path, index=False)

filled = roster_df['hometown'].notna().sum()
total  = len(roster_df)
print(f"\nHometown scraping complete.")
print(f"  Athletes with hometown: {filled}/{total} ({100*filled/total:.1f}%)")
print(f"  Unique hometowns found: {len(hometown_map)}")

## Section 13: Country Mapping (NEW)

Map hometown strings to WDI country codes. Handles US states, Canadian provinces, and international locations with fuzzy matching.

In [ ]:
# ============================================================
# COUNTRY MAPPING
# Maps hometown strings -> country_code (ISO 3-letter)
# ============================================================

# --- US State abbreviations (postal + AP style) ---
US_STATE_ABBREVS = {
    # Standard 2-letter postal codes
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY',
    'DC', 'PR', 'VI', 'GU', 'AS', 'MP',
}

US_STATE_NAMES = {
    'alabama', 'alaska', 'arizona', 'arkansas', 'california', 'colorado',
    'connecticut', 'delaware', 'florida', 'georgia', 'hawaii', 'idaho',
    'illinois', 'indiana', 'iowa', 'kansas', 'kentucky', 'louisiana',
    'maine', 'maryland', 'massachusetts', 'michigan', 'minnesota',
    'mississippi', 'missouri', 'montana', 'nebraska', 'nevada',
    'new hampshire', 'new jersey', 'new mexico', 'new york',
    'north carolina', 'north dakota', 'ohio', 'oklahoma', 'oregon',
    'pennsylvania', 'rhode island', 'south carolina', 'south dakota',
    'tennessee', 'texas', 'utah', 'vermont', 'virginia', 'washington',
    'west virginia', 'wisconsin', 'wyoming', 'district of columbia',
    'puerto rico',
}

# AP-style abbreviations (used in journalism / some rosters)
US_AP_ABBREVS = {
    'ala.', 'ariz.', 'ark.', 'calif.', 'colo.', 'conn.', 'del.', 'fla.',
    'ga.', 'ill.', 'ind.', 'kan.', 'ky.', 'la.', 'md.', 'mass.', 'mich.',
    'minn.', 'miss.', 'mo.', 'mont.', 'neb.', 'nev.', 'n.h.', 'n.j.',
    'n.m.', 'n.y.', 'n.c.', 'n.d.', 'okla.', 'ore.', 'pa.', 'r.i.',
    's.c.', 's.d.', 'tenn.', 'tex.', 'vt.', 'va.', 'wash.', 'w.va.',
    'wis.', 'wyo.',
}

# --- Canadian provinces ---
CANADIAN_PROVINCES = {
    'alberta', 'british columbia', 'manitoba', 'new brunswick',
    'newfoundland and labrador', 'newfoundland', 'nova scotia',
    'northwest territories', 'nunavut', 'ontario', 'prince edward island',
    'quebec', 'saskatchewan', 'yukon',
    # Abbreviations
    'ab', 'bc', 'mb', 'nb', 'nl', 'ns', 'nt', 'nu', 'on', 'pe', 'qc', 'sk', 'yt',
    # Common variants
    'b.c.', 'p.e.i.', 'n.b.', 'n.s.', 'n.l.', 'n.w.t.', 'ont.', 'que.', 'sask.', 'alta.', 'man.',
}

# --- Build WDI country name -> code mapping ---
# Use the macro DataFrame we already built
wdi_countries = macro[['country_code', 'country_name']].drop_duplicates()
wdi_country_dict = dict(zip(
    wdi_countries['country_name'].str.lower(),
    wdi_countries['country_code']
))
wdi_country_names = list(wdi_country_dict.keys())
wdi_country_codes = list(wdi_country_dict.values())


def map_hometown_to_country(hometown):
    """
    Map a hometown string to a WDI country code.

    Returns: (country_code, confidence) tuple
        confidence: 'exact', 'fuzzy', or 'unmatched'
    """
    if pd.isna(hometown) or not str(hometown).strip():
        return None, 'missing'

    hometown = str(hometown).strip()

    # Parse components: "City, Region" or "City, Region, Country"
    parts = [p.strip() for p in hometown.split(',')]

    if len(parts) >= 2:
        last_part = parts[-1].strip()
        last_lower = last_part.lower().strip('.')

        # Check if last part is a US state abbreviation
        if last_part.upper() in US_STATE_ABBREVS:
            return 'USA', 'exact'

        # Check if last part is a US state name
        if last_lower in US_STATE_NAMES:
            return 'USA', 'exact'

        # Check AP-style abbreviations
        if last_lower in US_AP_ABBREVS or (last_lower + '.') in US_AP_ABBREVS:
            return 'USA', 'exact'

        # Check Canadian provinces
        if last_lower in CANADIAN_PROVINCES:
            return 'CAN', 'exact'

        # Check if it is a known country name (exact match)
        if last_lower in wdi_country_dict:
            return wdi_country_dict[last_lower], 'exact'

        # Common country aliases
        country_aliases = {
            'united states': 'USA', 'usa': 'USA', 'us': 'USA',
            'u.s.': 'USA', 'u.s.a.': 'USA', 'america': 'USA',
            'canada': 'CAN', 'england': 'GBR', 'scotland': 'GBR',
            'wales': 'GBR', 'northern ireland': 'GBR',
            'great britain': 'GBR', 'uk': 'GBR', 'u.k.': 'GBR',
            'south korea': 'KOR', 'korea': 'KOR',
            'trinidad': 'TTO', 'trinidad and tobago': 'TTO',
            'ivory coast': 'CIV', "cote d'ivoire": 'CIV',
            'the bahamas': 'BHS', 'bahamas': 'BHS',
            'republic of ireland': 'IRL', 'ireland': 'IRL',
        }
        if last_lower in country_aliases:
            return country_aliases[last_lower], 'exact'

        # Fuzzy match against WDI country names
        result = process.extractOne(last_lower, wdi_country_names, scorer=fuzz.ratio)
        if result and result[1] >= 80:
            matched_name = result[0]
            return wdi_country_dict[matched_name], 'fuzzy'

    # Single-part hometown (just a city) -- assume US
    if len(parts) == 1:
        return 'USA', 'fuzzy'

    return None, 'unmatched'


print("Country mapping function loaded.")
print(f"  WDI country dictionary: {len(wdi_country_dict)} entries")
print(f"  US state abbrevs: {len(US_STATE_ABBREVS)}")
print(f"  Canadian provinces: {len(CANADIAN_PROVINCES)}")

# --- Apply mapping to roster data ---
print("\nMapping hometowns to country codes...")

# TFRRS roster
tfrrs_path = os.path.join(RAW_DATA_PATH, 'tfrrs_all_rosters.csv')
if os.path.exists(tfrrs_path):
    tfrrs_df = pd.read_csv(tfrrs_path)
    results = tfrrs_df['hometown'].apply(map_hometown_to_country)
    tfrrs_df['country_code'] = [r[0] for r in results]
    tfrrs_df['country_match_confidence'] = [r[1] for r in results]
    tfrrs_df.to_csv(tfrrs_path, index=False)
    print(f"  TFRRS: mapped {tfrrs_df['country_code'].notna().sum()}/{len(tfrrs_df)} athletes")
    print(f"  Confidence breakdown:")
    print(tfrrs_df['country_match_confidence'].value_counts().to_string())

# Soccer roster
if os.path.exists(SUCCESS_FILE):
    soccer_df = pd.read_csv(SUCCESS_FILE)
    results = soccer_df['hometown'].apply(map_hometown_to_country)
    soccer_df['country_code'] = [r[0] for r in results]
    soccer_df['country_match_confidence'] = [r[1] for r in results]
    soccer_df.to_csv(SUCCESS_FILE, index=False)
    print(f"\n  Soccer: mapped {soccer_df['country_code'].notna().sum()}/{len(soccer_df)} athletes")
    print(f"  Confidence breakdown:")
    print(soccer_df['country_match_confidence'].value_counts().to_string())

# Flag unmatched for review
unmatched_hometowns = set()
for path in [tfrrs_path, SUCCESS_FILE]:
    if os.path.exists(path):
        df_tmp = pd.read_csv(path)
        unmatched = df_tmp[df_tmp['country_match_confidence'] == 'unmatched']
        for ht in unmatched['hometown'].dropna().unique():
            unmatched_hometowns.add(ht)

if unmatched_hometowns:
    print(f"\nUnmatched hometowns ({len(unmatched_hometowns)} unique):")
    for ht in sorted(unmatched_hometowns)[:30]:
        print(f"  - {ht}")
    unmatched_path = os.path.join(RAW_DATA_PATH, 'unmatched_hometowns.csv')
    pd.DataFrame({'hometown': sorted(unmatched_hometowns)}).to_csv(unmatched_path, index=False)
    print(f"  Full list saved to: {unmatched_path}")

## Section 14: Merge Analysis Dataset (NEW)

Merge roster data with crosswalk, IPEDS, EADA, and macro indicators into a single analysis-ready dataset.

In [ ]:
# ============================================================
# BUILD ANALYSIS DATASET
# Merge: roster -> crosswalk -> IPEDS -> EADA -> macro
# ============================================================

# 1. Load roster data (both soccer and TFRRS)
roster_dfs = []

if os.path.exists(SUCCESS_FILE):
    soccer_df = pd.read_csv(SUCCESS_FILE)
    soccer_df['sport'] = 'Soccer'
    # Normalize column names to match TFRRS
    if 'name' in soccer_df.columns and 'athlete_name' not in soccer_df.columns:
        soccer_df.rename(columns={'name': 'athlete_name'}, inplace=True)
    roster_dfs.append(soccer_df)
    print(f"Soccer roster: {len(soccer_df)} athletes")

tfrrs_path = os.path.join(RAW_DATA_PATH, 'tfrrs_all_rosters.csv')
if os.path.exists(tfrrs_path):
    tfrrs_df = pd.read_csv(tfrrs_path)
    roster_dfs.append(tfrrs_df)
    print(f"TFRRS roster: {len(tfrrs_df)} athletes")

if not roster_dfs:
    raise ValueError("No roster data found!")

all_rosters = pd.concat(roster_dfs, ignore_index=True)
print(f"\nCombined roster: {len(all_rosters)} athletes")

# 2. Load crosswalk
crosswalk_path = os.path.join(RAW_DATA_PATH, 'ncaa_ipeds_crosswalk_verified.csv')
crosswalk = pd.read_csv(crosswalk_path)
print(f"Crosswalk: {len(crosswalk)} entries")

# 3. Join roster to crosswalk (school_name -> UNITID)
# Normalize school names for matching
all_rosters['school_name_clean'] = all_rosters['school_name'].apply(clean_text)
crosswalk['ncaa_name_clean'] = crosswalk['ncaa_name'].apply(clean_text)

merged = pd.merge(
    all_rosters,
    crosswalk[['ncaa_name_clean', 'ipeds_unitid', 'ncaa_division', 'ncaa_conference']],
    left_on='school_name_clean',
    right_on='ncaa_name_clean',
    how='left'
)
print(f"After crosswalk join: {len(merged)} rows, "
      f"{merged['ipeds_unitid'].notna().sum()} with UNITID")

# 4. Join to IPEDS institution data
hd_cols = ['UNITID', 'INSTNM', 'CITY', 'STABBR', 'SECTOR', 'CONTROL',
           'HLOFFER', 'OBEREG', 'LOCALE', 'INSTSIZE']
hd_subset = hd[[c for c in hd_cols if c in hd.columns]].copy()
hd_subset['UNITID'] = pd.to_numeric(hd_subset['UNITID'], errors='coerce')
merged['ipeds_unitid'] = pd.to_numeric(merged['ipeds_unitid'], errors='coerce')

merged = pd.merge(
    merged, hd_subset,
    left_on='ipeds_unitid', right_on='UNITID',
    how='left', suffixes=('', '_ipeds')
)
print(f"After IPEDS HD join: {len(merged)} rows")

# 5. Join to EADA (if loaded)
if eada is not None and len(eada) > 0:
    eada_subset = eada[['UNITID'] + [c for c in eada.columns if c != 'UNITID'][:10]].copy()
    eada_subset['UNITID'] = pd.to_numeric(eada_subset['UNITID'], errors='coerce')
    merged = pd.merge(
        merged, eada_subset,
        left_on='ipeds_unitid', right_on='UNITID',
        how='left', suffixes=('', '_eada')
    )
    print(f"After EADA join: {len(merged)} rows")

# 6. Join to macro indicators (country_code, year=2023)
if 'country_code' in merged.columns:
    macro_2023 = macro[macro['year'] == 2023].copy()
    macro_2023 = macro_2023.drop(columns=['year'], errors='ignore')

    merged = pd.merge(
        merged, macro_2023,
        on='country_code',
        how='left', suffixes=('', '_macro')
    )
    print(f"After macro join: {len(merged)} rows, "
          f"{merged['gdp_per_capita_ppp'].notna().sum()} with GDP data")

# 7. Save analysis dataset
output_path = os.path.join(RAW_DATA_PATH, 'analysis_dataset.csv')
merged.to_csv(output_path, index=False)
print(f"\nAnalysis dataset saved to: {output_path}")
print(f"Final shape: {merged.shape}")
print(f"Columns: {list(merged.columns)}")

## Section 15: Descriptive Statistics (NEW)

Exploratory analysis of international athlete representation, country distributions, and economic distance.

In [ ]:
# ============================================================
# DESCRIPTIVE STATISTICS
# ============================================================

# Load analysis dataset
analysis_path = os.path.join(RAW_DATA_PATH, 'analysis_dataset.csv')
df = pd.read_csv(analysis_path)
print(f"Analysis dataset: {df.shape[0]} athletes, {df.shape[1]} columns")

# Identify international athletes (not USA)
df['is_international'] = (df['country_code'].notna()) & (df['country_code'] != 'USA')
n_intl = df['is_international'].sum()
n_total = len(df)
print(f"\nInternational athletes: {n_intl} / {n_total} ({100*n_intl/n_total:.1f}%)")

# ---- 1. International athletes per school ----
print("\n--- International Athletes per School (Top 20) ---")
intl_by_school = df[df['is_international']].groupby('school_name').size().sort_values(ascending=False)
display(intl_by_school.head(20))

# ---- 2. International athletes per conference ----
if 'ncaa_conference' in df.columns:
    print("\n--- International Athletes per Conference (Top 15) ---")
    intl_by_conf = df[df['is_international']].groupby('ncaa_conference').size().sort_values(ascending=False)
    display(intl_by_conf.head(15))

# ---- 3. International athletes per division ----
if 'ncaa_division' in df.columns:
    print("\n--- International Athletes per Division ---")
    div_summary = df.groupby('ncaa_division').agg(
        total=('is_international', 'size'),
        international=('is_international', 'sum'),
    )
    div_summary['pct_international'] = 100 * div_summary['international'] / div_summary['total']
    display(div_summary.sort_values('pct_international', ascending=False))

In [ ]:
# ---- 4. Most represented countries ----
print("--- Most Represented Countries ---")
intl_df = df[df['is_international'] & df['country_code'].notna()].copy()
country_counts = intl_df['country_code'].value_counts()
print(f"\nTotal countries represented: {len(country_counts)}")
display(country_counts.head(25))

# ---- 5. Economic distance analysis ----
# Calculate GDP per capita PPP for the US as reference
us_gdp = macro[(macro['country_code'] == 'USA') & (macro['year'] == 2023)]['gdp_per_capita_ppp'].values
if len(us_gdp) > 0:
    us_gdp_val = us_gdp[0]
    print(f"\nUS GDP per capita PPP (2023): ${us_gdp_val:,.0f}")

    # Calculate economic distance for international athletes
    if 'gdp_per_capita_ppp' in intl_df.columns:
        intl_df = intl_df.copy()
        intl_df['econ_distance'] = us_gdp_val - intl_df['gdp_per_capita_ppp']
        intl_df['econ_distance_ratio'] = intl_df['gdp_per_capita_ppp'] / us_gdp_val

        print("\nEconomic Distance Summary (US GDP - Home GDP):")
        display(intl_df['econ_distance'].describe())

        # Average economic distance by country
        print("\nAverage Economic Distance by Country (Top 20 by athlete count):")
        top_countries = intl_df['country_code'].value_counts().head(20).index
        econ_by_country = intl_df[intl_df['country_code'].isin(top_countries)].groupby('country_code').agg(
            n_athletes=('athlete_name', 'count'),
            avg_gdp_ppp=('gdp_per_capita_ppp', 'mean'),
            avg_econ_distance=('econ_distance', 'mean'),
        ).sort_values('n_athletes', ascending=False)
        display(econ_by_country)
else:
    print("WARNING: Could not find US GDP data for 2023.")

In [ ]:
# ---- 6. Visualizations ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Top 15 countries by athlete count
ax1 = axes[0, 0]
top15 = country_counts.head(15)
top15.plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Top 15 Countries by International Athlete Count')
ax1.set_xlabel('Number of Athletes')
ax1.invert_yaxis()

# Plot 2: International athlete percentage by division
if 'ncaa_division' in df.columns:
    ax2 = axes[0, 1]
    div_pct = div_summary['pct_international'].sort_values(ascending=False)
    div_pct.plot(kind='bar', ax=ax2, color='coral')
    ax2.set_title('% International Athletes by Division')
    ax2.set_ylabel('Percentage')
    ax2.tick_params(axis='x', rotation=0)

# Plot 3: Distribution of economic distance
ax3 = axes[1, 0]
if 'econ_distance' in intl_df.columns:
    intl_df['econ_distance'].dropna().hist(bins=30, ax=ax3, color='seagreen', edgecolor='black')
    ax3.set_title('Distribution of Economic Distance (US GDP - Home GDP)')
    ax3.set_xlabel('Economic Distance (USD PPP)')
    ax3.set_ylabel('Count')

# Plot 4: GDP per capita vs enrollment count
ax4 = axes[1, 1]
if 'gdp_per_capita_ppp' in intl_df.columns:
    country_plot = intl_df.groupby('country_code').agg(
        n_athletes=('athlete_name', 'count'),
        avg_gdp=('gdp_per_capita_ppp', 'mean'),
    ).dropna()
    ax4.scatter(country_plot['avg_gdp'], country_plot['n_athletes'],
                alpha=0.6, color='darkorange', edgecolors='black', s=60)
    ax4.set_title('GDP per Capita PPP vs. Athlete Count by Country')
    ax4.set_xlabel('GDP per Capita PPP (USD)')
    ax4.set_ylabel('Number of Athletes')

    # Annotate top countries
    for code_val, row in country_plot.nlargest(5, 'n_athletes').iterrows():
        ax4.annotate(code_val, (row['avg_gdp'], row['n_athletes']),
                     fontsize=8, ha='left')

plt.tight_layout()
plt.savefig(os.path.join(BASE_PATH, 'output', 'descriptive_stats.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved to output/descriptive_stats.png")